# Klasifikasi Hate Speech dengan XLM-RoBERTa Base

## 1. Library

In [ ]:
import os
os.environ.update(USE_TF='0', USE_FLAX='0', TOKENIZERS_PARALLELISM='false')

import gc
import re
import hashlib
import unicodedata
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
from itertools import combinations
from importlib.metadata import version
from zipfile import ZipFile, ZIP_DEFLATED

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datasets import Dataset
from huggingface_hub import HfApi
from IPython.display import display, FileLink
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding,
    Trainer, TrainingArguments, EarlyStoppingCallback, set_seed,
)


In [ ]:
LANGUAGES = ['en', 'it', 'nl']
MODEL_NAME = 'FacebookAI/xlm-roberta-base'
SEED = 42
MAX_LENGTH = 384
EPOCHS = 10
PATIENCE = 3
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 16
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
LABELS = ['no', 'yes_implicit', 'yes_explicit']
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

In [ ]:
assert LANGUAGES and set(LANGUAGES) <= {'en', 'it', 'nl'}

set_seed(SEED)
print('GPU:', torch.cuda.get_device_name(0))
print('Bahasa:', ', '.join(LANGUAGES))

def cleanup():
    plt.close('all')
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

cleanup()

## 2. Baca dan periksa data ketiga bahasa

In [ ]:
DATA = {}
DATA_HASHES = {}
for DATA_LANGUAGE in LANGUAGES:
    splits = ('train', 'val', 'test')
    names = [f'{DATA_LANGUAGE}-{split}.csv' for split in splits]
    split_folder = f'{DATA_LANGUAGE}-splitting'
    candidates = [
        Path('.'), Path('data'), Path(split_folder),
        Path('data') / split_folder, Path('Analisis/data') / split_folder,
        Path('../data') / split_folder,
    ]

    for DATA_DIR in candidates:
        present = [(DATA_DIR / name).is_file() for name in names]
        if any(present):
            if not all(present):
                raise FileNotFoundError(f'Lengkapi {names} dalam folder {DATA_DIR.resolve()}.')
            break
    else:
        raise FileNotFoundError(f'Upload {names} ke {Path.cwd()}. Periksa DATA_LANGUAGE.')

    required = {'StereoQueerEval_id', 'yt_comment', 'hate_speech'}
    video_columns = {'yt_title', 'yt_description'}

    frames, source_hashes, audit_keys = {}, {}, {}
    normalize = lambda text: re.sub(r'\s+', ' ', unicodedata.normalize('NFKC', str(text)).casefold()).strip()

    for split, name in zip(splits, names):
        csv_path = DATA_DIR / name
        frame = pd.read_csv(csv_path, dtype=str, keep_default_na=False, encoding='utf-8-sig')

        missing = required - set(frame.columns)
        if missing:
            raise ValueError(f'{name}: kolom wajib tidak tersedia: {sorted(missing)}.')
        if frame.empty:
            raise ValueError(f'{name}: data kosong.')
        for column in ('StereoQueerEval_id', 'yt_comment'):
            if frame[column].str.strip().eq('').any():
                raise ValueError(f'{name}: {column} berisi nilai kosong.')
        if frame['StereoQueerEval_id'].duplicated().any():
            raise ValueError(f'{name}: terdapat ID duplikat.')
        if set(frame['hate_speech']) != set(LABELS):
            raise ValueError(f'{name}: kelas tidak sesuai dengan LABELS.')

        audit_keys[split] = {
            'id': set(frame['StereoQueerEval_id']),
            'comment': set(frame['yt_comment'].map(normalize)),
        }
        if video_columns <= set(frame.columns):
            contexts = zip(frame['yt_title'].map(normalize), frame['yt_description'].map(normalize))
            audit_keys[split]['video'] = {pair for pair in contexts if any(pair)}

        frames[split] = frame
        source_hashes[name] = hashlib.sha256(csv_path.read_bytes()).hexdigest()

    for left, right in combinations(frames, 2):
        shared_keys = audit_keys[left].keys() & audit_keys[right].keys()
        for key in shared_keys:
            if audit_keys[left][key] & audit_keys[right][key]:
                raise ValueError(f'Overlap {key}: {left} dan {right}. Periksa split asli.')

    print('Folder CSV:', DATA_DIR.resolve())
    class_counts = pd.DataFrame({split: frame['hate_speech'].value_counts() for split, frame in frames.items()}).T
    display(class_counts)
    DATA[DATA_LANGUAGE] = frames
    DATA_HASHES[DATA_LANGUAGE] = source_hashes

## 3. Siapkan perhitungan loss dan metrik

In [ ]:
def make_compute_loss(class_weights):
    """Ikat bobot kelas ke loss untuk satu eksperimen bahasa."""
    def compute_loss(outputs, labels, num_items_in_batch=None):
        return torch.nn.functional.cross_entropy(
            outputs.logits.float(), labels, weight=class_weights.to(outputs.logits.device),
        )

    return compute_loss

def compute_metrics(prediction):
    logits = prediction.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    predicted = np.asarray(logits).argmax(axis=-1)
    actual = prediction.label_ids
    return {
        'macro_f1': float(f1_score(actual, predicted, labels=range(len(LABELS)), average='macro', zero_division=0)),
        'accuracy': float(accuracy_score(actual, predicted)),
        'weighted_f1': float(f1_score(actual, predicted, labels=range(len(LABELS)), average='weighted', zero_division=0)),
    }

## 4. Pelatihan dan Evaluasi

In [ ]:
MODEL_COMMIT = HfApi().model_info(MODEL_NAME).sha
summaries = []

#### Persiapan dataset dan Trainer

In [ ]:
def prepare_datasets(frames, tokenizer):
    """Tokenisasi komentar asli dan pasangkan label untuk setiap split."""
    datasets = {}
    for split, frame in frames.items():
        tokens = tokenizer(frame['yt_comment'].tolist(), max_length=MAX_LENGTH, truncation=True)
        tokens['labels'] = frame['hate_speech'].map(LABEL2ID).tolist()
        datasets[split] = Dataset.from_dict(tokens)
    return datasets


def build_trainer(datasets, tokenizer, class_weights, out):
    """Muat model awal baru dan gunakan konfigurasi training yang sama."""
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, revision=MODEL_COMMIT, num_labels=len(LABELS),
        label2id=LABEL2ID, id2label=ID2LABEL,
    )
    model.config.use_cache = False

    args = TrainingArguments(
        output_dir=str(out / 'checkpoints'),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        lr_scheduler_type='linear',
        optim='adamw_torch',
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        eval_accumulation_steps=1,
        fp16=True,
        seed=SEED,
        data_seed=SEED,
        eval_strategy='epoch',
        logging_strategy='epoch',
        save_strategy='best',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        greater_is_better=True,
        save_total_limit=1,
        save_only_model=True,
        report_to='none',
        disable_tqdm=False,
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=datasets['train'],
        eval_dataset=datasets['val'],
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics,
        compute_loss_func=make_compute_loss(class_weights),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
    )
    trainer.model_accepts_loss_kwargs = False
    return trainer

#### Grafik pelatihan

In [ ]:
def save_training_history(trainer, out):
    """Simpan log, grafik loss, dan grafik macro-F1 validation."""
    history = pd.DataFrame(trainer.state.log_history)
    history.to_csv(out / 'training_log.csv', index=False)

    train_history = history.dropna(subset=['loss'])
    val_history = history.dropna(subset=['eval_loss'])
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(train_history['epoch'], train_history['loss'], marker='o', label='Train')
    ax.plot(val_history['epoch'], val_history['eval_loss'], marker='o', label='Validation')
    ax.set(xlabel='Epoch', ylabel='Loss', title='Loss pelatihan dan validation')
    ax.legend()
    fig.tight_layout()
    fig.savefig(out / 'training_loss.png', dpi=160)
    plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(val_history['epoch'], val_history['eval_macro_f1'], marker='o')
    ax.set(xlabel='Epoch', ylabel='Macro-F1', title='Macro-F1 validation')
    fig.tight_layout()
    fig.savefig(out / 'validation_macro_f1.png', dpi=160)
    plt.show()
    plt.close(fig)

#### Evaluasi validation dan test

In [ ]:
def evaluate_split(trainer, dataset, frame, split, out):
    """Evaluasi satu split dan simpan metrik, prediksi, serta confusion matrix."""
    prediction = trainer.predict(dataset, metric_key_prefix=split)
    logits = prediction.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    predicted = np.asarray(logits).argmax(axis=-1)
    actual = prediction.label_ids
    metrics = compute_metrics(prediction)
    metrics['loss'] = float(prediction.metrics[f'{split}_loss'])

    report = pd.DataFrame(classification_report(
        actual, predicted, labels=range(len(LABELS)), target_names=LABELS,
        output_dict=True, zero_division=0,
    )).T
    report.to_csv(out / f'{split}_report.csv')
    print(split.upper(), metrics)
    display(report)

    predictions = frame[['StereoQueerEval_id', 'yt_comment', 'hate_speech']].copy()
    predictions['predicted_label'] = [ID2LABEL[int(i)] for i in predicted]
    predictions['correct'] = predictions['hate_speech'] == predictions['predicted_label']
    predictions.to_csv(out / f'{split}_prediction.csv', index=False)

    fig, ax = plt.subplots(figsize=(7, 5))
    matrix = ConfusionMatrixDisplay.from_predictions(
        actual, predicted, labels=range(len(LABELS)), display_labels=LABELS,
        cmap='Blues', colorbar=False, ax=ax,
    )
    ax.set_title(f'Confusion matrix {split}')
    fig.tight_layout()
    fig.savefig(out / f'{split}_confusion_matrix.png', dpi=160)
    pd.DataFrame(matrix.confusion_matrix, index=LABELS, columns=LABELS).to_csv(
        out / f'{split}_confusion_matrix.csv', index_label='actual_label',
    )
    plt.show()
    plt.close(fig)
    return metrics

#### Penyimpanan hasil

In [ ]:
def save_results(data_language, out, trainer, train_result, class_weights, results):
    """Simpan konfigurasi dan metrik, buat ZIP CSV/PNG, lalu commit ke Modal."""
    config = {
        'model': MODEL_NAME, 'revision': MODEL_COMMIT, 'data_language': data_language,
        'input': 'yt_comment', 'preprocessing': 'raw', 'labels': LABELS,
        'label_to_id': LABEL2ID, 'max_length': MAX_LENGTH, 'seed': SEED,
        'loss': 'weighted_cross_entropy', 'class_weights': dict(zip(LABELS, class_weights.tolist())),
        'training': trainer.args.to_dict(), 'early_stopping_patience': PATIENCE,
        'best_validation_macro_f1': trainer.state.best_metric,
        'best_model_checkpoint': trainer.state.best_model_checkpoint,
        'train_metrics': train_result.metrics, 'csv_sha256': DATA_HASHES[data_language],
        'versions': {name: version(name) for name in ('torch', 'transformers', 'accelerate', 'datasets')},
        'model_exported': False,
    }
    config_rows = []
    for key, value in config.items():
        if isinstance(value, dict):
            config_rows.extend({'parameter': f'{key}.{name}', 'value': item} for name, item in value.items())
        else:
            config_rows.append({'parameter': key, 'value': value})
    pd.DataFrame(config_rows).to_csv(out / 'config.csv', index=False)
    pd.DataFrame(results).T.to_csv(out / 'metrics.csv')

    zip_path = out.with_suffix('.zip')
    with ZipFile(zip_path, 'w', compression=ZIP_DEFLATED) as archive:
        for file in out.rglob('*'):
            if file.is_file() and file.suffix.lower() in {'.csv', '.png'} and file.relative_to(out).parts[0] != 'checkpoints':
                archive.write(file, file.relative_to(out.parent))
    if out.resolve().is_relative_to(Path('/workspace').resolve()):
        import modal
        modal.Volume.from_name('ptm-qwen').commit()
        print('Hasil dan ZIP tersimpan di Volume ptm-qwen.')

#### Alur eksperimen satu bahasa

In [ ]:
def run_language(data_language):
    """Jalankan satu eksperimen bahasa dengan model dan bobot kelas terpisah."""
    cleanup()
    set_seed(SEED)
    frames = DATA[data_language]
    out = Path('results') / f'xlm_roberta_base_data-{data_language}_{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}_{uuid4().hex[:8]}'
    out.mkdir(parents=True, exist_ok=False)
    print('Bahasa:', data_language, '| Hasil:', out)

    trainer = tokenizer = datasets = class_weights = train_result = None
    try:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_COMMIT, use_fast=True)
        datasets = prepare_datasets(frames, tokenizer)
        class_counts = frames['train']['hate_speech'].value_counts().reindex(LABELS)
        class_weights = torch.tensor(
            len(frames['train']) / (len(LABELS) * class_counts.to_numpy()), dtype=torch.float32,
        )
        print('Bobot kelas:', dict(zip(LABELS, class_weights.tolist())))

        trainer = build_trainer(datasets, tokenizer, class_weights, out)
        train_result = trainer.train()
        print('Macro-F1 validation terbaik:', trainer.state.best_metric)
        save_training_history(trainer, out)

        results = {}
        for split in ('val', 'test'):
            results[split] = evaluate_split(trainer, datasets[split], frames[split], split, out)
            cleanup()
        save_results(data_language, out, trainer, train_result, class_weights, results)
        summaries.append({
            'language': data_language,
            'validation_macro_f1': results['val']['macro_f1'],
            'test_macro_f1': results['test']['macro_f1'],
            'output': str(out),
        })
    finally:
        trainer = tokenizer = datasets = class_weights = train_result = None
        cleanup()

### 4.1. English


In [ ]:
run_language('en')

### 4.2. Italian


In [ ]:
run_language('it')

### 4.3. Dutch NL


In [ ]:
run_language('nl')

## 5. Ringkasan hasil

In [ ]:
display(pd.DataFrame(summaries))